# DBAASP-dev: pull every peptide
Pages the list endpoint for the authoritative id list, then fetches the full record (`/peptides/{id}`: SMILES, MIC, targets, ...) for each id. Raw JSON is cached in `.cache/dbaasp_dev_raw.jsonl`, so re-runs and interrupted runs resume without re-fetching.

In [1]:
import json, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

from soamp.curation.dbaasp_client import make_session

BASE = "https://dbaasp-dev.ibrsp.org/peptides"
HEADERS = {"accept": "application/json"}
PAGE = 500          # server rejects limit=5000; 500 works
MAX_RPS = 5         # global request cap; 8 workers uncapped ran at ~30/s and got 403-blocked after ~1.7k requests
MAX_WORKERS = 8     # dev server 403-blocked us at 50 workers (prod tolerated ~11k requests at 50, then failed too)
MAX_RETRIES = 3
BLOCK_WAIT_S = 300       # dev 403-blocks after ~1.5k requests; the block has cleared within minutes-to-half-an-hour
MAX_STALLED_CYCLES = 6   # give up after this many consecutive wait cycles with zero new peptides

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_PATH = ROOT / ".cache" / "dbaasp_dev_raw.jsonl"
IDS_PATH = ROOT / ".cache" / "dbaasp_dev_ids.json"  # cached id list + totalCount, so re-runs need no network
RAW_PATH.parent.mkdir(exist_ok=True)
session = make_session()

## 1. Id list and total count

In [2]:
def list_all_summaries():
    rows, offset, total = [], 0, None
    while total is None or offset < total:
        for attempt in range(MAX_RETRIES):
            r = session.get(BASE, params={"limit": PAGE, "offset": offset}, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                break
            time.sleep(1 + attempt)
        r.raise_for_status()
        body = r.json()
        total = body["totalCount"]
        rows += body["data"]
        offset += PAGE
    return rows, total

if IDS_PATH.exists():
    cached_list = json.loads(IDS_PATH.read_text())
    total_count, ids = cached_list["total_count"], cached_list["ids"]
else:
    summaries, total_count = list_all_summaries()
    ids = sorted({s["id"] for s in summaries})
    assert len(ids) == len(summaries) == total_count, (len(ids), len(summaries), total_count)
    IDS_PATH.write_text(json.dumps({"total_count": total_count, "ids": ids}))
print(f"DBAASP-dev totalCount = {total_count:,} peptides (ids {ids[0]}..{ids[-1]})")

DBAASP-dev totalCount = 25,892 peptides (ids 1..26162)


## 2. Fetch full records (resumable)

In [3]:
def load_cached_ids():
    done = set()
    if RAW_PATH.exists():
        with open(RAW_PATH) as f:
            for line in f:
                try:
                    done.add(json.loads(line)["id"])
                except (ValueError, KeyError):
                    pass  # truncated last line from an interrupted run
    return done

_rate_lock = threading.Lock()
_next_slot = [0.0]

def throttle():
    # reserve the next 1/MAX_RPS time slot (shared by all workers), then sleep until it
    with _rate_lock:
        now = time.monotonic()
        slot = max(now, _next_slot[0])
        _next_slot[0] = slot + 1 / MAX_RPS
    time.sleep(max(0.0, slot - now))

blocked = threading.Event()  # set on a 403 so every worker stops hitting the server

def fetch_full(peptide_id):
    last_err = None
    for attempt in range(MAX_RETRIES):
        if blocked.is_set():
            return peptide_id, None, "skipped (blocked)"
        throttle()
        try:
            r = session.get(f"{BASE}/{peptide_id}", headers=HEADERS, timeout=30)
            if r.status_code == 200:
                return peptide_id, r.json(), None
            if r.status_code == 403:  # rate-limit block from the load balancer; retrying only extends it
                blocked.set()
                return peptide_id, None, "http 403"
            last_err = f"http {r.status_code}"
        except (requests.RequestException, ValueError) as e:
            last_err = str(e)
        time.sleep(0.5 * (attempt + 1))
    return peptide_id, None, last_err

def pull_batch(todo):
    """Fetch until done or the server 403s. Returns (n_fetched, errors)."""
    n, errs = 0, {}
    with open(RAW_PATH, "a") as out, ThreadPoolExecutor(MAX_WORKERS) as pool:
        futures = [pool.submit(fetch_full, i) for i in todo]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            pid, payload, err = fut.result()
            if payload is None:
                errs[pid] = err
            else:
                out.write(json.dumps(payload) + "\n")
                out.flush()
                n += 1
    return n, errs

stalled = 0
errors = {}
while True:
    cached = load_cached_ids()  # read the cache once per cycle, not once per id
    todo = [i for i in ids if i not in cached]
    print(f"{len(ids) - len(todo):,} cached, {len(todo):,} to fetch")
    if not todo:
        break
    blocked.clear()
    n_fetched, errors = pull_batch(todo)
    stalled = 0 if n_fetched else stalled + 1
    if blocked.is_set():
        if stalled >= MAX_STALLED_CYCLES:
            print(f"No progress for {stalled} cycles; giving up. Re-run later, it resumes from the cache.")
            break
        print(f"BLOCKED (HTTP 403) after {n_fetched:,} new peptides; waiting {BLOCK_WAIT_S}s then resuming "
              f"(stalled cycles: {stalled}/{MAX_STALLED_CYCLES})")
        time.sleep(BLOCK_WAIT_S)
    elif not n_fetched:
        break  # only non-403 errors left; reported below

real_errors = {k: v for k, v in errors.items() if not v.startswith(("skipped", "http 403"))}
print(f"done. cached = {len(load_cached_ids()):,}/{len(ids):,}; non-403 errors: {len(real_errors)}")


25,892 cached, 0 to fetch


done. cached = 25,892/25,892; non-403 errors: 0


## 3. DataFrames

In [4]:
raw = {}
with open(RAW_PATH) as f:
    for line in f:
        try:
            rec = json.loads(line)
        except ValueError:
            continue
        raw[rec["id"]] = rec
missing = set(ids) - set(raw)
assert not missing, f"{len(missing)} peptides not fetched yet; re-run the fetch cell"

def seq_full(rec):
    if rec.get("sequence"):
        return rec["sequence"]
    return "/".join(m.get("sequence") or "" for m in rec.get("monomers") or [])

def name_of(x):
    return x.get("name") if isinstance(x, dict) else x

df_peptides = pd.DataFrame([
    {
        "id": r["id"],
        "dbaaspId": r.get("dbaaspId"),
        "name": r.get("name"),
        "sequence_full": seq_full(r),
        "sequenceLength": r.get("sequenceLength"),
        "complexity": r.get("complexity"),
        "synthesisType": r.get("synthesisType"),
        "nTerminus": r.get("nTerminus"),
        "cTerminus": r.get("cTerminus"),
        "smiles": [s["smiles"] for s in (r.get("smiles") or []) if s.get("smiles")],
        "n_unusual_aa": len(r.get("unusualAminoAcids") or []),
        "n_target_activities": len(r.get("targetActivities") or []),
        "n_hemolytic": len(r.get("hemoliticCytotoxicActivities") or []),
        "n_antibiofilm": len(r.get("antibiofilmActivities") or []),
    }
    for r in raw.values()
]).sort_values("id").reset_index(drop=True)
df_peptides["has_smiles"] = df_peptides["smiles"].str.len() > 0

act_rows = []
for r in raw.values():
    for a in r.get("targetActivities") or []:
        act_rows.append({
            "peptide_id": r["id"],
            "activity_id": a.get("id"),
            "target_species": name_of(a.get("targetSpecies")),
            "measure": name_of(a.get("activityMeasureGroup")) or a.get("activityMeasureValue"),
            "activity": a.get("activity"),
            "concentration": a.get("concentration"),
            "unit": name_of(a.get("unit")),
            "medium": name_of(a.get("medium")),
            "cfu": a.get("cfu"),
            "ph": a.get("ph"),
            "ionicStrength": a.get("ionicStrength"),
            "saltType": a.get("saltType"),
            "note": a.get("note"),
        })
df_activities = pd.DataFrame(act_rows)

# whole dataframe: one row per (peptide, activity); peptides with no activity keep one row
df_full = df_peptides.merge(df_activities, how="left", left_on="id", right_on="peptide_id")

assert len(df_peptides) == total_count
assert df_activities["peptide_id"].isin(df_peptides["id"]).all()
print(f"df_peptides   {df_peptides.shape}\ndf_activities {df_activities.shape}\ndf_full       {df_full.shape}")
print(f"native SMILES: {df_peptides['has_smiles'].mean():.1%} of peptides; "
      f"peptides with activity rows: {(df_peptides['n_target_activities'] > 0).mean():.1%}")
print(df_activities["measure"].value_counts().head(8))
df_full.head()

df_peptides   (25892, 15)
df_activities (199901, 13)
df_full       (200566, 28)
native SMILES: 97.6% of peptides; peptides with activity rows: 97.4%
measure
MIC      144144
MBC       14116
IC50      10384
-          6849
           5000
LC         2155
MIC50      1764
MIC90      1684
Name: count, dtype: int64


,id,dbaaspId,name,sequence_full,sequenceLength,complexity,synthesisType,nTerminus,cTerminus,smiles,...,measure,activity,concentration,unit,medium,cfu,ph,ionicStrength,saltType,note
0,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,MIC,0.261290,14.5,µM,TSB,,,,,
1,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,MIC,0.504560,28.0,µM,TSB,,,,,
2,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,MIC,0.504560,28.0,µM,TSB,,,,,
3,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,MIC,0.261290,14.5,µM,TSB,,,,,
4,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,MIC,0.210474,0.73-11.68,µM,MHB,5E5,,,,10 Clinical isolates


## 3b. Organism column (strains merged to species, same as prod)

In [5]:
from soamp.curation.taxonomy import classify_species  # same helper prod step 06 uses

# organism = first two words of target_species (genus + epithet), exactly as in
# pipeline/curation/06_assay_filter_units.py; domain/taxid come from the same call
unique_targets = df_activities["target_species"].dropna().unique()
tax = {name: classify_species(name) for name in unique_targets}  # name -> (domain, taxid, binomial)

for df in (df_activities, df_full):
    t = df["target_species"].map(tax)
    df["organism"] = t.map(lambda x: x[2] if isinstance(x, tuple) else None)
    df["organism_domain"] = t.map(lambda x: x[0] if isinstance(x, tuple) else None)
    df["ncbi_taxon_id"] = t.map(lambda x: x[1] if isinstance(x, tuple) else None).astype("Int64")

# target_kind: what the target actually is. NCBI domain where it resolved; for domain == "Unknown"
# (first two words aren't an NCBI genus) fall back to name patterns for viruses and host cells.
VIRUS_RE = (r"HIV|HSV|SARS|MERS|HCoV|virus|Hepatitis|DENV|Influenza|immunodeficiency|syncytial|Stomatitis|"
            r"enteritis|Zika|West Nile|Rous sarcoma|Marburg|Chikungunya|encephalitis|Foot-and-Mouth|"
            r"gastroenteritis|Coxsackie|Porcine (reproductive|epidemic)")
HOST_RE = (r"^(Human|Mouse|Murine|Rat|Hamster|Bovine|Pig|Rabbit|Monkey|Canine)\b|Jurkat|HeLa|cancer|"
           r"myeloid|Ehrlich|carcinoma|lymphoma|leukemia|melanoma|glioma")

def target_kind(domain, name):
    if not isinstance(name, str):
        return None
    if domain != "Unknown":
        return str(domain).lower()
    if pd.Series([name]).str.contains(VIRUS_RE, case=False, regex=True).iloc[0]:
        return "virus"  # checked first so "Human Influenza" is a virus, not a host cell
    if pd.Series([name]).str.contains(HOST_RE, case=False, regex=True).iloc[0]:
        return "host_cell"
    return "unclassified"

kind = {name: target_kind(tax[name][0], name) for name in unique_targets}
for df in (df_activities, df_full):
    df["target_kind"] = df["target_species"].map(kind)

print(df_activities["target_kind"].value_counts(dropna=False).to_string())
print(f"{len(unique_targets):,} raw target_species -> {df_activities['organism'].nunique():,} organisms")
print(df_activities["organism_domain"].value_counts(dropna=False).to_string())
df_activities["organism"].value_counts().head(10)

/var/folders/wg/3tpsh2513fz9vs64nsklg0sc0000gn/T/ipykernel_37358/1794960323.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if pd.Series([name]).str.contains(VIRUS_RE, case=False, regex=True).iloc[0]:
/var/folders/wg/3tpsh2513fz9vs64nsklg0sc0000gn/T/ipykernel_37358/1794960323.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if pd.Series([name]).str.contains(HOST_RE, case=False, regex=True).iloc[0]:


target_kind
bacteria        163123
fungus           21000
host_cell        10410
virus             4028
other              987
animal             248
unclassified        75
archaea             16
plant               13
NaN                  1
8,319 raw target_species -> 1,369 organisms
organism_domain
Bacteria    163123
Fungus       21000
Unknown      14444
Other          987
Animal         248
Virus           69
Archaea         16
Plant           13
NaN              1


organism
Staphylococcus aureus         33913
Escherichia coli              32692
Pseudomonas aeruginosa        20906
Candida albicans               8362
Klebsiella pneumoniae          8006
Acinetobacter baumannii        7227
Bacillus subtilis              6534
Enterococcus faecalis          5244
Staphylococcus epidermidis     5179
Salmonella enterica            5102
Name: count, dtype: int64

## 4. Save the united DataFrame

In [6]:
OUT_DIR = ROOT / "data"
parquet_path = OUT_DIR / "dbaasp_dev_full.parquet"  # keeps the `smiles` list column as a real list
csv_path = OUT_DIR / "dbaasp_dev_full.csv"          # readable copy; `smiles` stored as a JSON string

df_full.to_parquet(parquet_path, index=False)
df_full.assign(smiles=df_full["smiles"].map(json.dumps)).to_csv(csv_path, index=False)

# round-trip check so we know the saved file is the whole frame
back = pd.read_parquet(parquet_path)
assert back.shape == df_full.shape and back["id"].nunique() == total_count
for pth in (parquet_path, csv_path):
    print(f"{pth.relative_to(ROOT)}  {pth.stat().st_size / 1e6:.1f} MB")

data/dbaasp_dev_full.parquet  14.5 MB
data/dbaasp_dev_full.csv  227.9 MB


In [36]:
import pandas as pd

df = pd.read_parquet("/Users/lukajin/PycharmProjects/soamp/data/dbaasp_dev_full.parquet")

In [37]:
df["activity"].describe()

count    1.999010e+05
mean     6.596991e+04
std      2.623837e+05
min      0.000000e+00
25%      7.669880e+00
50%      2.831361e+01
75%      1.193584e+02
max      1.111111e+06
Name: activity, dtype: float64

In [8]:
df.head()

,id,dbaaspId,name,sequence_full,sequenceLength,complexity,synthesisType,nTerminus,cTerminus,smiles,...,medium,cfu,ph,ionicStrength,saltType,note,organism,organism_domain,ncbi_taxon_id,target_kind
0,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,TSB,,,,,,Escherichia coli,Bacteria,562,bacteria
1,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,TSB,,,,,,Staphylococcus aureus,Bacteria,1280,bacteria
2,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,TSB,,,,,,Pseudomonas aeruginosa,Bacteria,287,bacteria
3,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,TSB,,,,,,Enterococcus faecalis,Bacteria,1351,bacteria
4,1,DBAASPR_1,Distinctin,ENREVPPGFTALIKTLRKCKII/NLVSGLIEARKYLEQLHRKLKNCKV,0.0,"{'description': '', 'name': 'Multimer'}",{'name': 'Ribosomal'},None,None,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,...,MHB,5E5,,,,10 Clinical isolates,Escherichia coli,Bacteria,562,bacteria


In [38]:
df[df["dbaaspId"] == "DBAASPR_37"]

,id,dbaaspId,name,sequence_full,sequenceLength,complexity,synthesisType,nTerminus,cTerminus,smiles,...,medium,cfu,ph,ionicStrength,saltType,note,organism,organism_domain,ncbi_taxon_id,target_kind
303,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,MHB,5E5,,,,,Escherichia coli,Bacteria,562,bacteria
304,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,MHB,5E5,,,,,Staphylococcus aureus,Bacteria,1280,bacteria
305,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,MHB,5E5,,,,,Pseudomonas aeruginosa,Bacteria,287,bacteria
306,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,7.2±1.9 - IC50 for amphotericin B,Kluyveromyces marxianus,Fungus,4911,fungus
307,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,2.2±0.8 - IC50 for amphotericin B,Candida albicans,Fungus,5476,fungus
308,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,1.0±0.1 - IC50 for amphotericin B,Cryptococcus neoformans,Fungus,5207,fungus
309,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,>70 - IC50 for amphotericin B,Candida krusei,Fungus,4909,fungus
310,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,6.0±1.0 - IC50 for amphotericin B,Candida parapsilosis,Fungus,5475,fungus
311,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,1.9±0.2 - IC50 for amphotericin B,Candida glabrata,Fungus,5478,fungus
312,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,"{'description': None, 'name': 'Monomer'}",{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,SDA,,,,,1) 4.6±0.1 - IC50 for amphotericin B. 2) Fluco...,Candida glabrata,Fungus,5478,fungus


In [9]:
df["has_smiles"].dtype

dtype('bool')

In [10]:
columns = ["peptide_id", "dbaaspId", "name", "smiles", "target_kind", "target_species", "organism", "organism_domain", "ncbi_taxon_id", "measure", "activity", "concentration", "unit", "note"]

data = df.loc[df["has_smiles"], columns]

In [11]:
data

,peptide_id,dbaaspId,name,smiles,target_kind,target_species,organism,organism_domain,ncbi_taxon_id,measure,activity,concentration,unit,note
0,1.0,DBAASPR_1,Distinctin,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,bacteria,Escherichia coli ATCC 25922,Escherichia coli,Bacteria,562,MIC,0.261290,14.5,µM,
1,1.0,DBAASPR_1,Distinctin,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,bacteria,Staphylococcus aureus ATCC 25923,Staphylococcus aureus,Bacteria,1280,MIC,0.504560,28.0,µM,
2,1.0,DBAASPR_1,Distinctin,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,bacteria,Pseudomonas aeruginosa ATCC 27853,Pseudomonas aeruginosa,Bacteria,287,MIC,0.504560,28.0,µM,
3,1.0,DBAASPR_1,Distinctin,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,bacteria,Enterococcus faecalis ATCC 29212,Enterococcus faecalis,Bacteria,1351,MIC,0.261290,14.5,µM,
4,1.0,DBAASPR_1,Distinctin,[CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...,bacteria,Escherichia coli,Escherichia coli,Bacteria,562,MIC,0.210474,0.73-11.68,µM,10 Clinical isolates
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198035,25812.0,DBAASPR_25812,Cathelicidin MC-CATH3,[CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...,bacteria,Shigella flexneri ATCC 12022,Shigella flexneri,Bacteria,623,MIC,9.380000,9.38,µg/ml,
198036,25812.0,DBAASPR_25812,Cathelicidin MC-CATH3,[CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...,fungus,Candida glabrata,Candida glabrata,Fungus,5478,MIC,100.000000,>100,µg/ml,Clinical isolate
198037,25812.0,DBAASPR_25812,Cathelicidin MC-CATH3,[CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...,fungus,Candida albicans 08090102,Candida albicans,Fungus,5476,MIC,18.750000,18.75,µg/ml,
198038,25812.0,DBAASPR_25812,Cathelicidin MC-CATH3,[CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...,host_cell,Murine macrophage cells RAW 264.7,Murine macrophage,Unknown,<NA>,10-20% Cytotoxicity,319.614070,64,µM,


In [12]:
df.shape, data.shape

((200566, 32), (196594, 14))

In [13]:
df['peptide_id'].nunique(), data['peptide_id'].nunique()

(25227, 24649)

In [14]:
data["activity"].value_counts()

activity
1.111111e+06    11442
3.200000e+01     5821
8.000000e+00     5731
1.600000e+01     5599
1.280000e+02     5342
                ...  
4.067610e+02        1
2.033805e+02        1
1.641645e+02        1
3.283290e+02        1
2.728112e+01        1
Name: count, Length: 48455, dtype: int64

In [15]:
data[data["target_species"].notna()].count()

peptide_id         195970
dbaaspId           195970
name               195970
smiles             195970
target_kind        195970
target_species     195970
organism           195970
organism_domain    195970
ncbi_taxon_id      181865
measure            195970
activity           195970
concentration      195970
unit               184474
note               193159
dtype: int64

In [16]:
data["smiles"]

0         [CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...
1         [CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...
2         [CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...
3         [CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...
4         [CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCC...
                                ...                        
198035    [CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...
198036    [CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...
198037    [CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...
198038    [CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...
198039    [CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)...
Name: smiles, Length: 196594, dtype: object

## 5. Murcko-scaffold clusters for E. coli / S. aureus / P. aeruginosa
Select the three organisms, dedupe peptides by SMILES, then cluster by Bemis-Murcko scaffold (peptides sharing a scaffold = one cluster; no ring = `no_scaffold`).

In [17]:
import re
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")
THREE_ORGS = ["Escherichia coli", "Staphylococcus aureus", "Pseudomonas aeruginosa"]

df_3org_rows = df_full[df_full["organism"].isin(THREE_ORGS)].copy()      # activity-level rows
peps = df_peptides[df_peptides["id"].isin(df_3org_rows["id"].unique())].copy()
print(f"{len(df_3org_rows):,} activity rows, {len(peps):,} unique peptides")
print(df_3org_rows.groupby("organism")["id"].nunique().to_string())

[2026-11-06 14:13:49,877] Enabling RDKit 2026.03.5 jupyter extensions


87,511 activity rows, 20,783 unique peptides
organism
Escherichia coli          17992
Pseudomonas aeruginosa    12684
Staphylococcus aureus     17150


### 5a. One fixed tautomer per peptide
DBAASP lists one SMILES per tautomer of ambiguous residues (Histidine HID/HIE, Arginine C1/C2). They are different RDKit molecules, so we pick by label: Histidine HID and Arginine C1.

In [18]:
def pick_variant(rec):
    """(smiles, n_variants, rule). Skip Histidine tautomers other than HID and Arginine tautomers other than C1."""
    cands = [s for s in rec.get("smiles") or [] if s.get("smiles")]
    if not cands:
        return None, 0, "no_smiles"
    keep = []
    for s in cands:
        d = re.sub(r"\s+", " ", s.get("description") or "")
        if re.search(r"Histidine: (?!HID)\w", d) or re.search(r"Arginine: (?!C1)\w", d):
            continue
        keep.append(s)
    if len(cands) == 1:
        return cands[0]["smiles"], 1, "single"
    if len(keep) == 1:
        return keep[0]["smiles"], len(cands), "label"
    return cands[0]["smiles"], len(cands), "fallback_first"   # none or several matched: should not happen

picked = {pid: pick_variant(raw[pid]) for pid in peps["id"]}
peps["smiles_fixed"] = peps["id"].map(lambda i: picked[i][0])
peps["n_variants"] = peps["id"].map(lambda i: picked[i][1])
peps["variant_rule"] = peps["id"].map(lambda i: picked[i][2])
print(peps["variant_rule"].value_counts().to_string())
assert (peps["variant_rule"] != "fallback_first").all(), "label rule did not resolve every multi-variant peptide"

variant_rule
label        12789
single        7614
no_smiles      380


### 5b. Compare and dedupe peptides with identical SMILES

In [19]:
has_smiles = peps[peps["smiles_fixed"].notna()].copy()
print(f"{len(peps) - len(has_smiles):,} peptides without SMILES are excluded from clustering")

orgs_of = df_3org_rows.groupby("id")["organism"].agg(lambda s: sorted(set(s)))
has_smiles["organisms"] = has_smiles["id"].map(orgs_of)
has_smiles["synthesis"] = has_smiles["synthesisType"].map(name_of)

groups = has_smiles.groupby("smiles_fixed")["id"].agg(list)
dup_groups = groups[groups.map(len) > 1]
print(f"{len(dup_groups)} SMILES shared by >1 peptide id: {int(dup_groups.map(len).sum())} ids, "
      f"{int((dup_groups.map(len) - 1).sum())} removable")

dup_table = (has_smiles[has_smiles["smiles_fixed"].isin(dup_groups.index)]
             .sort_values(["smiles_fixed", "id"])
             [["smiles_fixed", "id", "name", "sequence_full", "synthesis", "nTerminus", "cTerminus", "organisms"]]
             .reset_index(drop=True))
dup_table.insert(0, "dup_group", dup_table["smiles_fixed"].factorize()[0])
dup_table["smiles_fixed"] = dup_table["smiles_fixed"].str[:40] + "..."
diff_fields = {f: int(dup_table.groupby("dup_group")[f].agg(lambda s: s.astype(str).nunique() > 1).sum())
               for f in ["name", "sequence_full", "synthesis", "nTerminus", "cTerminus"]}
print("groups where the field differs between duplicates:", diff_fields)
dup_table

380 peptides without SMILES are excluded from clustering


24 SMILES shared by >1 peptide id: 53 ids, 29 removable
groups where the field differs between duplicates: {'name': 17, 'sequence_full': 13, 'synthesis': 4, 'nTerminus': 0, 'cTerminus': 0}


,dup_group,smiles_fixed,id,name,sequence_full,synthesis,nTerminus,cTerminus,organisms
0,0,C=C1NC(=O)[C@H](CCCCN)NC(=O)[C@@H](NC(=O...,14990,Nisin-P,VXSKXLCTPGCKTGILMXCAIKTATCGCHFG,Ribosomal,None,None,"[Escherichia coli, Pseudomonas aeruginosa, Sta..."
1,0,C=C1NC(=O)[C@H](CCCCN)NC(=O)[C@@H](NC(=O...,21783,Lantibiotic Agalacticin,VXSKXLCTPGCKTGILMXCAIKTATCGCHFG,Ribosomal,None,None,"[Pseudomonas aeruginosa, Staphylococcus aureus]"
2,1,CC(=O)N(O)CCC[C@@H]1NC(=O)[C@H](CCCN(O)C...,22176,Acremonpeptide D,XXXWLF,Nonribosomal,None,None,"[Escherichia coli, Staphylococcus aureus]"
3,1,CC(=O)N(O)CCC[C@@H]1NC(=O)[C@H](CCCN(O)C...,22177,Al(III) - Acremonpeptide D,XXXWLF,Nonribosomal,None,None,"[Escherichia coli, Staphylococcus aureus]"
4,2,CC(C)C[C@@H](NC(=O)CNC(=O)[C@H](N)Cc1c[n...,10731,"WMR, Myxinidin [H3,6l;D4R;P11R]",wGlrrllkyGkrs,Synthetic,None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[Pseudomonas aeruginosa]
5,2,CC(C)C[C@@H](NC(=O)CNC(=O)[C@H](N)Cc1c[n...,25324,"D-W-Myxinidin-3,4,11R;2,4L",wglrrllkygkrs,Synthetic,None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[Staphylococcus aureus]
6,3,CC(C)C[C@@H]1NC(=O)[C@H](CCCN=C(N)N)NC(=...,3338,"Protegrin-1-mimetic, Cyclic L8-1",LRLKKRRWKYRVpP,Synthetic,None,None,"[Escherichia coli, Pseudomonas aeruginosa, Sta..."
7,3,CC(C)C[C@@H]1NC(=O)[C@H](CCCN=C(N)N)NC(=...,23269,,RWKYRVpPLRLKKR,Synthetic,None,None,"[Escherichia coli, Pseudomonas aeruginosa, Sta..."
8,4,CC(C)C[C@H](NC(=O)CNC(=O)[C@H](CCCCN)NC(...,6672,,GKGLXXFKXXKFXXFKXXKFXXFKXXFKR,Synthetic,None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[Staphylococcus aureus]
9,4,CC(C)C[C@H](NC(=O)CNC(=O)[C@H](CCCCN)NC(...,10805,,GKGLXXFKXXKFXXFKXXKFXXFKXXFKR,Synthetic,None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[Staphylococcus aureus]


In [20]:
kept = has_smiles.sort_values("id").drop_duplicates("smiles_fixed", keep="first").copy()   # lowest id represents the group
rep_of = {i: grp[0] for grp in groups for i in grp}
kept["duplicate_ids"] = kept["smiles_fixed"].map(lambda s: groups[s][1:])
df_3org_rows["peptide_id_rep"] = df_3org_rows["id"].map(rep_of).fillna(df_3org_rows["id"]).astype(int)
assert kept["smiles_fixed"].is_unique
print(f"{len(has_smiles):,} peptides with SMILES -> {len(kept):,} after SMILES dedupe")

20,403 peptides with SMILES -> 20,374 after SMILES dedupe


### 5c. Bemis-Murcko scaffold = cluster

In [21]:
def murcko(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None                                   # parse failure
    scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    return scaf if scaf else "no_scaffold"            # acyclic peptide: no ring system

kept["murcko_scaffold"] = kept["smiles_fixed"].map(murcko)
df_parse_fail = kept[kept["murcko_scaffold"].isna()]
print(f"RDKit parse failures excluded: {len(df_parse_fail)}")
kept = kept[kept["murcko_scaffold"].notna()].copy()

sizes = kept["murcko_scaffold"].value_counts()
rank = {s: i for i, s in enumerate(sorted(sizes.index, key=lambda s: (-sizes[s], s)))}   # 0 = largest, ties by scaffold text
kept["murcko_cluster_id"] = kept["murcko_scaffold"].map(rank).astype(int)
kept["murcko_cluster_size"] = kept["murcko_scaffold"].map(sizes).astype(int)

print(f"{len(kept):,} peptides -> {len(sizes):,} Murcko clusters; singletons: {(sizes == 1).sum():,}")
print(f"no_scaffold cluster: {sizes.get('no_scaffold', 0):,} peptides")
print("size buckets (clusters):", pd.cut(sizes, [0, 1, 2, 5, 10, 50, 10**6], labels=["1", "2", "3-5", "6-10", "11-50", ">50"]).value_counts().sort_index().to_dict())
sizes.head(10).rename("top-10 cluster sizes").to_frame()

RDKit parse failures excluded: 0
20,374 peptides -> 6,529 Murcko clusters; singletons: 4,642
no_scaffold cluster: 2,140 peptides
size buckets (clusters): {'1': 4642, '2': 817, '3-5': 644, '6-10': 222, '11-50': 175, '>50': 29}


,top-10 cluster sizes
murcko_scaffold,
no_scaffold,2140
c1ccccc1,1146
c1ccc2[nH]ccc2c1,919
C1CCNC1,371
c1c[nH]cn1,233
O=C(CCc1c[nH]c2ccccc12)NCCc1c[nH]c2ccccc12,133
O=C(CCc1ccccc1)NCC(=O)N1CCCC1,120
O=C(CNC(=O)C1CCCN1)NC(Cc1ccccc1)C(=O)NCC(=O)N1CCCC1C(=O)NCC(=O)N1CCCC1C(=O)NCC(=O)N1CCCC1C(=O)N1CCCC1C(=O)NC(Cc1cnc[nH]1)C(=O)N1CCCC1,108
O=C(CCc1ccccc1)NCC(=O)NCC(=O)NCC(=O)NCC(=O)NCC(=O)NCC(=O)NCC(=O)NCC(=O)NCC(=O)NCCc1ccccc1,103


### 5d. Final DataFrame and save

In [22]:
df_peptide_clusters = (kept
    .assign(n_activity_rows_3org=lambda d: d["id"].map(df_3org_rows.groupby("peptide_id_rep").size()).fillna(0).astype(int))
    .drop(columns=["synthesis"])
    .sort_values(["murcko_cluster_id", "id"]).reset_index(drop=True))

assert df_peptide_clusters["murcko_cluster_size"].groupby(df_peptide_clusters["murcko_cluster_id"]).first().sum() == len(df_peptide_clusters)
assert df_peptide_clusters["murcko_cluster_id"].notna().all() and df_peptide_clusters["smiles_fixed"].is_unique

cluster_path = ROOT / "data" / "dbaasp_dev_3org_murcko_clusters.parquet"
df_peptide_clusters.to_parquet(cluster_path, index=False)
back = pd.read_parquet(cluster_path)
assert back.shape == df_peptide_clusters.shape
print(f"saved {cluster_path.relative_to(ROOT)}  {cluster_path.stat().st_size / 1e6:.1f} MB  shape {back.shape}")
df_peptide_clusters.head()

saved data/dbaasp_dev_3org_murcko_clusters.parquet  4.4 MB  shape (20374, 24)


,id,dbaaspId,name,sequence_full,sequenceLength,complexity,synthesisType,nTerminus,cTerminus,smiles,...,has_smiles,smiles_fixed,n_variants,variant_rule,organisms,duplicate_ids,murcko_scaffold,murcko_cluster_id,murcko_cluster_size,n_activity_rows_3org
0,37,DBAASPR_37,PGLa,GMASKAGAIAGKIAKVALKAL,21.0,{'name': 'Monomer'},{'name': 'Ribosomal'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C...,...,True,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C@...,1,single,"[Escherichia coli, Pseudomonas aeruginosa, Sta...",[],no_scaffold,0,2140,21
1,51,DBAASPS_51,,KLlK,4.0,{'name': 'Monomer'},{'name': 'Synthetic'},"{'description': 'Palmitic acid, Hexadecanoic a...","{'description': 'Amide, NH2', 'molarMass': 0.0...",[CCCCCCCCCCCCCCCC(=O)N[C@@H](CCCCN)C(=O)N[C@@H...,...,True,CCCCCCCCCCCCCCCC(=O)N[C@@H](CCCCN)C(=O)N[C@@H]...,1,single,"[Escherichia coli, Pseudomonas aeruginosa, Sta...",[],no_scaffold,0,2140,4
2,90,DBAASPS_90,Dermaseptin S4 (4-15)AMD[M4K],KTLLKKVLKAAA,12.0,{'name': 'Monomer'},{'name': 'Synthetic'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N...,...,True,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC...,1,single,"[Escherichia coli, Pseudomonas aeruginosa, Sta...",[],no_scaffold,0,2140,3
3,91,DBAASPS_91,NC7- Dermaseptin S4 (4-15)AMD[M4K],XKTLLKKVLKAAA,13.0,{'name': 'Monomer'},{'name': 'Synthetic'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N...,...,True,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC...,1,single,"[Escherichia coli, Pseudomonas aeruginosa, Sta...",[],no_scaffold,0,2140,3
4,92,DBAASPS_92,NC12- Dermaseptin S4 (4-15)AMD[M4K],XKTLLKKVLKAAA,13.0,{'name': 'Monomer'},{'name': 'Synthetic'},None,"{'description': 'Amide, NH2', 'molarMass': 0.0...",[CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N...,...,True,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC...,1,single,"[Escherichia coli, Pseudomonas aeruginosa, Sta...",[],no_scaffold,0,2140,3


### 5e. Ring systems only (linkers dropped)
Plain Murcko keeps the backbone atoms that link two rings, so peptides with the same rings but different spacing get different scaffolds. Here the cluster key is just the set of ring systems (fused rings count as one system), ignoring the linkers. Chirality and generic scaffolds were also tried and changed under 2% of the grouping, so they are not added.

In [23]:
def ring_system_key(scaffold_smiles):
    """Sorted ring systems of a Murcko scaffold, joined with '|'. Fused rings (sharing a bond) = one system."""
    if scaffold_smiles == "no_scaffold":
        return "no_scaffold"
    mol = Chem.MolFromSmiles(scaffold_smiles)
    systems = []
    for ring in mol.GetRingInfo().AtomRings():
        ring = set(ring)
        for s in [s for s in systems if len(s & ring) >= 2]:
            systems.remove(s)
            ring |= s
        systems.append(ring)
    parts = sorted(Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(s), isomericSmiles=False) for s in systems)
    return "|".join(parts) or "no_scaffold"

df_peptide_clusters["ringsys_key"] = df_peptide_clusters["murcko_scaffold"].map(ring_system_key)
rs_sizes = df_peptide_clusters["ringsys_key"].value_counts()
rs_rank = {k: i for i, k in enumerate(sorted(rs_sizes.index, key=lambda k: (-rs_sizes[k], k)))}   # 0 = largest
df_peptide_clusters["ringsys_cluster_id"] = df_peptide_clusters["ringsys_key"].map(rs_rank).astype(int)
df_peptide_clusters["ringsys_cluster_size"] = df_peptide_clusters["ringsys_key"].map(rs_sizes).astype(int)

def summarise(col):
    sz = df_peptide_clusters[col].value_counts()
    ringed = sz.drop("no_scaffold", errors="ignore")
    return {"clusters": len(sz), "singletons": int((sz == 1).sum()),
            "ring-free group": int(sz.get("no_scaffold", 0)), "largest ring-bearing cluster": int(ringed.max())}

comparison = pd.DataFrame({"plain Murcko (current)": summarise("murcko_scaffold"),
                           "ring systems only": summarise("ringsys_key")})
print(comparison.to_string())

multi = df_peptide_clusters[df_peptide_clusters["ringsys_key"].str.contains("|", regex=False)]
print(f"\n{len(multi):,} peptides have 2+ ring systems: {multi['murcko_scaffold'].nunique():,} distinct plain scaffolds "
      f"-> {multi['ringsys_key'].nunique():,} distinct ring-system combinations")
comparison

                              plain Murcko (current)  ring systems only
clusters                                        6529               2458
singletons                                      4642               1415
ring-free group                                 2140               2140
largest ring-bearing cluster                    1146               1146

14,882 peptides have 2+ ring systems: 6,361 distinct plain scaffolds -> 2,290 distinct ring-system combinations


,plain Murcko (current),ring systems only
clusters,6529,2458
singletons,4642,1415
ring-free group,2140,2140
largest ring-bearing cluster,1146,1146


In [24]:
assert df_peptide_clusters[["murcko_cluster_id", "ringsys_cluster_id"]].notna().all().all()
df_peptide_clusters.to_parquet(cluster_path, index=False)
back = pd.read_parquet(cluster_path)
assert back.shape == df_peptide_clusters.shape
print(f"saved {cluster_path.relative_to(ROOT)}  {cluster_path.stat().st_size / 1e6:.1f} MB  shape {back.shape}")

saved data/dbaasp_dev_3org_murcko_clusters.parquet  4.5 MB  shape (20374, 27)
